# Fe-Cr Binary Alloy — FEM Nanoindentation Analysis
## Oliver-Pharr Reduced Modulus Extraction

**Pipeline:** DFT (QE 7.3.1) → GP/MLP Surrogate → CalculiX CCX 2.21 FEM → Oliver-Pharr

**System:** Fe₁₆Cr₀ to Fe₀Cr₁₆ BCC binary alloy, single-crystal anisotropic elastic constants  
**Indenter:** Conical, 70.3° half-included angle, diamond (E=1141 GPa, ν=0.07)  
**Units:** µm / µN / MPa throughout  
**Target output:** Reduced modulus Eᵣ vs Cr% composition curve

---

## 0. Dependencies

In [ ]:
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.optimize import curve_fit
from scipy.signal import savgol_filter
from pathlib import Path

plt.rcParams.update({
    'font.family': 'serif',
    'font.size': 11,
    'axes.labelsize': 12,
    'axes.titlesize': 12,
    'legend.fontsize': 10,
    'figure.dpi': 150,
})

## 1. Configuration

Set `.dat` file paths and composition metadata here.

In [ ]:
# ── Composition registry ──────────────────────────────────────────────────────
# Keys: composition label
# 'dat'     : path to CCX .dat output file
# 'cr_pct'  : Cr atomic percent (n_cr / 16 * 100)
# 'n_cr'    : number of Cr atoms in 16-atom BCC supercell
# 'tier'    : DFT convergence tier (A=1e-8/1e-7, B=1e-7, C=1e-5 AFM)
# 'C11','C12','C44' : elastic constants used in FEM (MPa, from best ML surrogate)

COMPOSITIONS = {
    'Fe16Cr0': {
        'dat'   : Path('../fem/conical/fe16cr00/indentation_fe16cr00_v16.dat'),
        'cr_pct': 0.0,
        'n_cr'  : 0,
        'tier'  : 'A',
        'C11'   : 307430,
        'C12'   : 160930,
        'C44'   : 119690,
    },
    'Fe8Cr8': {
        'dat'   : Path('../fem/conical/fe08cr08/indentation_fe08cr08_v16.dat'),
        'cr_pct': 50.0,
        'n_cr'  : 8,
        'tier'  : 'A',
        'C11'   : 300500,
        'C12'   : 164470,
        'C44'   : 123170,
    },
    'Fe4Cr12': {
        'dat'   : Path('../fem/conical/fe04cr12/indentation_fe04cr12_v16.dat'),
        'cr_pct': 75.0,
        'n_cr'  : 12,
        'tier'  : 'B',
        'C11'   : 396510,
        'C12'   : 174770,
        'C44'   : 115440,
    },
    'Fe0Cr16': {
        'dat'   : Path('../fem/conical/fe00cr16/indentation_fe00cr16_v16.dat'),
        'cr_pct': 100.0,
        'n_cr'  : 16,
        'tier'  : 'C',
        'C11'   : 438290,
        'C12'   :  66060,
        'C44'   : 100200,
    },
}

# ── Indenter geometry ─────────────────────────────────────────────────────────
THETA_DEG   = 70.3          # conical half-included angle (degrees)
THETA_RAD   = np.radians(THETA_DEG)
EPSILON     = 0.727         # geometric correction for conical indenter (Sneddon)

# ── Indenter material (diamond) ───────────────────────────────────────────────
E_IND       = 1141000.0     # MPa
NU_IND      = 0.07

# ── Oliver-Pharr fit window ───────────────────────────────────────────────────
# Fraction of the unloading curve to include in the power-law fit
# Standard practice: top 25–95% of unloading load range
UNLOAD_FIT_UPPER = 0.95     # upper bound as fraction of P_max
UNLOAD_FIT_LOWER = 0.25     # lower bound as fraction of P_max

# ── Unit conversion ───────────────────────────────────────────────────────────
# FEM units: µm, µN, MPa
# P(h) curve kept in µN / µm for Oliver-Pharr; Er reported in GPa
MPA_TO_GPA  = 1e-3

## 2. Parser — CCX `.dat` → P(h) curve

Extracts `(h, P)` pairs from the `total force` and per-node displacement blocks.  
U2 from node 10 (first node in `nset_ind_top`, all nodes carry identical prescribed displacement).  
RF2 total from `total force` line.  
Gap of 0.01 µm subtracted so h=0 corresponds to first substrate contact.

In [ ]:
def parse_dat(filepath: Path) -> pd.DataFrame:
    """
    Parse a CCX .dat file produced by v16 input deck.

    Returns a DataFrame with columns:
        time    : CCX normalised time (0→1 loading, 1→1.1 unloading)
        h_raw   : U2 of node 10 in nset_ind_top (µm, negative = into substrate)
        h       : indentation depth (µm, positive, gap-corrected: h = |h_raw| - 0.01)
        P       : total RF2 on nset_ind_top (µN, positive = compressive load)
        phase   : 'loading' or 'unloading'
    """
    content = filepath.read_text()

    # ── Total RF2 at each time increment ─────────────────────────────────────
    pat_rf = (
        r'total force \(fx,fy,fz\) for set NSET_IND_TOP and time'
        r'\s+([\d.E+-]+)\s*\n\s*\n\s+[\d.E+-]+\s+([\d.E+-]+)'
    )
    rf_matches = re.findall(pat_rf, content)
    rf_data = {float(t): float(fy) for t, fy in rf_matches}

    # ── U2 of node 10 at each time increment ─────────────────────────────────
    pat_u2 = (
        r'displacements \(vx,vy,vz\) for set NSET_IND_TOP and time'
        r'\s+([\d.E+-]+)\s*\n\s*\n\s+10\s+[\d.E+-]+\s+([\d.E+-]+)'
    )
    u2_matches = re.findall(pat_u2, content)
    u2_data = {float(t): float(vy) for t, vy in u2_matches}

    # ── Align on common time keys ─────────────────────────────────────────────
    times = sorted(set(rf_data) & set(u2_data))
    records = []
    for t in times:
        h_raw = u2_data[t]               # µm, negative
        P_raw = rf_data[t]               # µN, negative (compressive)
        h     = abs(h_raw) - 0.01        # gap-corrected depth (µm)
        P     = abs(P_raw)               # positive load convention
        phase = 'loading' if t <= 1.0 else 'unloading'
        records.append({'time': t, 'h_raw': h_raw, 'h': h, 'P': P, 'phase': phase})

    df = pd.DataFrame(records)
    # remove pre-contact noise (h < 0 → gap not yet closed)
    df = df[df['h'] >= 0].reset_index(drop=True)
    return df


# Parse all compositions
ph_data = {}
for label, cfg in COMPOSITIONS.items():
    ph_data[label] = parse_dat(cfg['dat'])
    df = ph_data[label]
    h_max = df['h'].max()
    P_max = df.loc[df['phase'] == 'loading', 'P'].max()
    n_load = (df['phase'] == 'loading').sum()
    n_unload = (df['phase'] == 'unloading').sum()
    print(f"{label:10s}  h_max={h_max:.4f} µm  P_max={P_max:.2f} µN  "
          f"n_load={n_load}  n_unload={n_unload}")

## 3. P(h) Curve Visualisation

In [ ]:
colors = {
    'Fe16Cr0' : '#1f4e79',
    'Fe8Cr8'  : '#2e75b6',
    'Fe4Cr12' : '#70ad47',
    'Fe0Cr16' : '#c00000',
}

fig, ax = plt.subplots(figsize=(7, 5))

for label, df in ph_data.items():
    cr = COMPOSITIONS[label]['cr_pct']
    load   = df[df['phase'] == 'loading']
    unload = df[df['phase'] == 'unloading']
    c = colors[label]
    ax.plot(load['h'],   load['P'],   color=c, lw=1.8,
            label=f'{label} ({cr:.0f}% Cr)')
    ax.plot(unload['h'], unload['P'], color=c, lw=1.8, ls='--')

ax.set_xlabel('Indentation depth h (µm)')
ax.set_ylabel('Load P (µN)')
ax.set_title('P(h) curves — Fe-Cr conical nanoindentation (FEM)')
ax.legend()
ax.set_xlim(left=0)
ax.set_ylim(bottom=0)
plt.tight_layout()
plt.savefig('../fem/post/fecr_ph_curves.png', dpi=200)
plt.show()

## 4. Oliver-Pharr Analysis

### Method

The Oliver-Pharr method extracts reduced modulus from the initial unloading slope:

$$S = \left.\frac{dP}{dh}\right|_{h_{\max}} = \frac{2}{\sqrt{\pi}} E_r \sqrt{A_c}$$

For a **conical indenter** (Sneddon contact), the projected contact area is:

$$A_c = \pi h_c^2 \tan^2\theta$$

where the contact depth is:

$$h_c = h_{\max} - \epsilon \frac{P_{\max}}{S}, \quad \epsilon = 0.727 \text{ (conical)}$$

The unloading curve is fit with a power law over the top 25–95% of the unloading load range:

$$P = A(h - h_f)^m$$

The stiffness $S = dP/dh|_{h_{\max}}$ is obtained analytically from this fit.  
Reduced modulus is then:

$$E_r = \frac{S \sqrt{\pi}}{2 \sqrt{A_c}}$$

The indenter compliance correction gives the substrate modulus:

$$\frac{1}{E_r} = \frac{1 - \nu_s^2}{E_s} + \frac{1 - \nu_i^2}{E_i}$$

Note: $\nu_s$ for anisotropic single crystals requires an effective Poisson's ratio. For BCC Fe-Cr an orientation-averaged value is used (see cell 5).

In [ ]:
def power_law(h, A, hf, m):
    return A * (h - hf) ** m


def oliver_pharr(df: pd.DataFrame,
                 theta_rad: float,
                 epsilon: float,
                 fit_upper: float = 0.95,
                 fit_lower: float = 0.25) -> dict:
    """
    Applies the Oliver-Pharr method to a parsed P(h) DataFrame.

    Parameters
    ----------
    df         : DataFrame from parse_dat()
    theta_rad  : conical indenter half-angle in radians
    epsilon    : geometric correction factor (0.727 for cone)
    fit_upper  : upper P/P_max fraction for power-law fit window
    fit_lower  : lower P/P_max fraction for power-law fit window

    Returns
    -------
    dict with keys: P_max, h_max, S, hf, hc, Ac, Er_MPa, fit_params, fit_df
    """
    loading = df[df['phase'] == 'loading'].copy()
    unloading = df[df['phase'] == 'unloading'].copy()

    # Peak values at end of loading
    P_max = loading['P'].max()
    h_max = loading.loc[loading['P'].idxmax(), 'h']

    # Fit window: top fraction of unloading by load
    mask = (
        (unloading['P'] <= fit_upper * P_max) &
        (unloading['P'] >= fit_lower * P_max)
    )
    fit_df = unloading[mask].copy()

    if len(fit_df) < 5:
        raise ValueError(
            f'Insufficient unloading points in fit window: {len(fit_df)}. '
            'Adjust fit_upper / fit_lower bounds.'
        )

    # Initial guess for hf: residual depth ~ 0.05 * h_max
    hf0 = 0.05 * h_max
    p0 = [P_max, hf0, 1.5]
    bounds = ([0, -h_max, 0.5], [1e9, h_max, 4.0])

    popt, pcov = curve_fit(
        power_law,
        fit_df['h'].values,
        fit_df['P'].values,
        p0=p0,
        bounds=bounds,
        maxfev=10000,
    )
    A_fit, hf_fit, m_fit = popt

    # Stiffness S = dP/dh at h_max
    S = A_fit * m_fit * (h_max - hf_fit) ** (m_fit - 1)

    # Contact depth and projected area (conical)
    hc = h_max - epsilon * P_max / S
    Ac = np.pi * hc**2 * np.tan(theta_rad)**2

    # Reduced modulus (µN/µm² = MPa)
    Er_MPa = (S * np.sqrt(np.pi)) / (2.0 * np.sqrt(Ac))

    return {
        'P_max'      : P_max,
        'h_max'      : h_max,
        'S'          : S,
        'hf'         : hf_fit,
        'hc'         : hc,
        'Ac'         : Ac,
        'Er_MPa'     : Er_MPa,
        'fit_params' : {'A': A_fit, 'hf': hf_fit, 'm': m_fit},
        'fit_cov'    : pcov,
        'fit_df'     : fit_df,
        'n_fit_pts'  : len(fit_df),
    }


# Run Oliver-Pharr on all compositions
op_results = {}
for label, df in ph_data.items():
    try:
        result = oliver_pharr(
            df,
            theta_rad=THETA_RAD,
            epsilon=EPSILON,
            fit_upper=UNLOAD_FIT_UPPER,
            fit_lower=UNLOAD_FIT_LOWER,
        )
        op_results[label] = result
        print(
            f"{label:10s}  P_max={result['P_max']:.2f} µN  "
            f"h_max={result['h_max']:.4f} µm  "
            f"S={result['S']:.2f} µN/µm  "
            f"hc={result['hc']:.4f} µm  "
            f"Er={result['Er_MPa']*MPA_TO_GPA:.2f} GPa  "
            f"m={result['fit_params']['m']:.3f}  "
            f"n_pts={result['n_fit_pts']}"
        )
    except Exception as e:
        print(f"{label:10s}  ERROR: {e}")

## 5. Indenter Compliance Correction

Eᵣ contains contributions from both substrate and indenter.  
The substrate modulus Eₛ (effective, orientation-averaged) is recovered via:

$$\frac{1}{E_r} = \frac{1 - \nu_s^2}{E_s} + \frac{1 - \nu_i^2}{E_i}$$

For a BCC anisotropic crystal, an effective isotropic Poisson's ratio νₛ is estimated  
from the Hill average of the elastic constants. This is an approximation —  
the true contact response integrates over multiple crystallographic orientations.

In [ ]:
def hill_poisson(C11, C12, C44):
    """
    Voigt-Reuss-Hill average Poisson's ratio for a cubic crystal.
    Returns nu_VRH (dimensionless).
    Reference: Hill (1952), Proc. Phys. Soc. A.
    """
    # Voigt bulk and shear moduli
    Kv = (C11 + 2*C12) / 3.0
    Gv = (C11 - C12 + 3*C44) / 5.0

    # Reuss bulk and shear moduli
    S11 = (C11 + C12) / ((C11 - C12) * (C11 + 2*C12))
    S12 = -C12 / ((C11 - C12) * (C11 + 2*C12))
    S44 = 1.0 / C44
    Kr  = 1.0 / (3*(S11 + 2*S12))
    Gr  = 5.0 / (4*(S11 - S12) + 3*S44)

    # Hill averages
    K_hill = (Kv + Kr) / 2.0
    G_hill = (Gv + Gr) / 2.0

    nu = (3*K_hill - 2*G_hill) / (2*(3*K_hill + G_hill))
    return nu


def substrate_modulus(Er_MPa, nu_s, E_ind=E_IND, nu_ind=NU_IND):
    """
    Recover substrate Young's modulus from reduced modulus.
    Returns Es_MPa.
    """
    inv_Er  = 1.0 / Er_MPa
    inv_Ei  = (1.0 - nu_ind**2) / E_ind
    Es_MPa  = (1.0 - nu_s**2) / (inv_Er - inv_Ei)
    return Es_MPa


# Compute Er and Es for all compositions
summary_rows = []
for label, result in op_results.items():
    cfg  = COMPOSITIONS[label]
    nu_s = hill_poisson(cfg['C11'], cfg['C12'], cfg['C44'])
    Er   = result['Er_MPa']
    Es   = substrate_modulus(Er, nu_s)
    summary_rows.append({
        'label'   : label,
        'cr_pct'  : cfg['cr_pct'],
        'n_cr'    : cfg['n_cr'],
        'tier'    : cfg['tier'],
        'C11_GPa' : cfg['C11'] * MPA_TO_GPA,
        'C12_GPa' : cfg['C12'] * MPA_TO_GPA,
        'C44_GPa' : cfg['C44'] * MPA_TO_GPA,
        'nu_s'    : round(nu_s, 4),
        'P_max_uN': round(result['P_max'], 3),
        'h_max_um': round(result['h_max'], 5),
        'S_uN_um' : round(result['S'], 3),
        'hc_um'   : round(result['hc'], 5),
        'Er_GPa'  : round(Er  * MPA_TO_GPA, 2),
        'Es_GPa'  : round(Es  * MPA_TO_GPA, 2),
        'm_exp'   : round(result['fit_params']['m'], 4),
    })

summary_df = pd.DataFrame(summary_rows).set_index('label')
display(summary_df)

## 6. Unloading Fit Quality Check

Inspect the power-law fit for each composition. The fit should track the upper portion of the unloading curve closely. Poor fit quality (residuals > ~5%) should be flagged before accepting the Eᵣ value.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 8))
axes = axes.flatten()

for ax, (label, result) in zip(axes, op_results.items()):
    df    = ph_data[label]
    cfg   = COMPOSITIONS[label]
    unload = df[df['phase'] == 'unloading']
    fit_df = result['fit_df']

    # Full unloading curve
    ax.plot(unload['h'], unload['P'],
            color='#404040', lw=1.5, label='FEM unloading')

    # Fit window data points
    ax.scatter(fit_df['h'], fit_df['P'],
               color=colors[label], s=8, zorder=3, label='fit window')

    # Power-law fit curve
    h_fit = np.linspace(fit_df['h'].min(), result['h_max'], 200)
    p = result['fit_params']
    P_fit = power_law(h_fit, p['A'], p['hf'], p['m'])
    ax.plot(h_fit, P_fit,
            color=colors[label], lw=1.5, ls='--',
            label=f"fit: m={p['m']:.3f}")

    # Annotate S and Er
    Er_GPa = result['Er_MPa'] * MPA_TO_GPA
    ax.set_title(
        f"{label} ({cfg['cr_pct']:.0f}% Cr, tier {cfg['tier']})\n"
        f"S={result['S']:.1f} µN/µm   Eᵣ={Er_GPa:.2f} GPa"
    )
    ax.set_xlabel('h (µm)')
    ax.set_ylabel('P (µN)')
    ax.legend(fontsize=8)

plt.suptitle('Oliver-Pharr unloading fit quality', y=1.01)
plt.tight_layout()
plt.savefig('../fem/post/fecr_op_fits.png', dpi=200, bbox_inches='tight')
plt.show()

## 7. Eᵣ vs Cr% — Main Result

In [ ]:
cr_pct = summary_df['cr_pct'].values
Er_vals = summary_df['Er_GPa'].values
Es_vals = summary_df['Es_GPa'].values
tiers   = summary_df['tier'].values

tier_markers = {'A': 'o', 'B': 's', 'C': '^'}
tier_labels  = {'A': 'Tier A (conv_thr ≤1e-7)', 'B': 'Tier B (conv_thr 1e-7)', 'C': 'Tier C (conv_thr 1e-5, AFM)'}

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, (yvals, ylabel, title) in zip(
    axes,
    [
        (Er_vals, 'Reduced modulus Eᵣ (GPa)', 'Reduced modulus vs Cr content'),
        (Es_vals, 'Substrate modulus Eₛ (GPa)', 'Substrate modulus vs Cr content'),
    ]
):
    plotted_tiers = set()
    for x, y, t, lbl in zip(cr_pct, yvals, tiers, summary_df.index):
        mk = tier_markers[t]
        tl = tier_labels[t] if t not in plotted_tiers else None
        ax.scatter(x, y, marker=mk, s=80, color=colors[lbl],
                   zorder=5, label=tl)
        ax.annotate(lbl, (x, y), textcoords='offset points',
                    xytext=(6, 4), fontsize=8)
        plotted_tiers.add(t)

    ax.plot(cr_pct, yvals, color='#888888', lw=1.0, ls='-', zorder=1)
    ax.set_xlabel('Cr content (at%)')
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.legend(fontsize=8)
    ax.set_xlim(-5, 105)

plt.suptitle(
    'Fe-Cr binary alloy — DFT→ML→FEM nanoindentation\n'
    'Conical indenter 70.3°, depth 1.0 µm, single-crystal anisotropic BCC',
    y=1.02
)
plt.tight_layout()
plt.savefig('../fem/post/fecr_Er_vs_Cr.png', dpi=200, bbox_inches='tight')
plt.show()

print('\nFinal result table:')
display(summary_df[['cr_pct','tier','Er_GPa','Es_GPa','m_exp','nu_s']].rename(columns={
    'cr_pct':'Cr%','tier':'DFT tier','Er_GPa':'Eᵣ (GPa)',
    'Es_GPa':'Eₛ (GPa)','m_exp':'m (power law)','nu_s':'νₛ (Hill avg)'
}))

## 8. GP Uncertainty Bands on Eᵣ

GP posterior standard deviation is taken from `fem_material_inputs_best.csv` (ML phase, `analysis/` on Azog).  
That file contains MLP mean predictions (used as FEM input) and GP posterior std per constant per composition.  

Uncertainty in Eᵣ is propagated numerically: Oliver-Pharr is re-evaluated at ±1σ perturbed elastic constants,  
treating C11, C12, C44 uncertainties as independent. The resulting Eᵣ spread is reported as ±ΔEᵣ.

**Columns expected in `fem_material_inputs_best.csv`:**  
`composition, C11_GPa, C12_GPa, C44_GPa, std_C11_GPa, std_C12_GPa, std_C44_GPa`

The composition column must match the keys in `COMPOSITIONS` (e.g. `Fe16Cr0`, `Fe8Cr8`, etc.).

In [ ]:
# ── Load GP posterior std from fem_material_inputs_best.csv (ML phase) ───────
# File location on Azog: ~/dft_projects/fe-cr_cu-ni_DFT/analysis/fem_material_inputs_best.csv
# Copy to data/ alongside this notebook before running
FEM_INPUTS_CSV = Path('../analysis/fem_material_inputs_best.csv')


def er_sensitivity(df_ph: pd.DataFrame,
                   C11_MPa: float, C12_MPa: float, C44_MPa: float,
                   dC11: float, dC12: float, dC44: float,
                   theta_rad: float, epsilon: float) -> dict:
    """
    Numerical ±1σ sensitivity of Eᵣ to perturbations in C11, C12, C44.

    For each constant Cij, Oliver-Pharr is re-evaluated with the Hill-averaged
    Poisson's ratio recomputed at C±σ. Contact geometry (h_max, P_max, S)
    is taken from the FEM result — it is independent of the material constants
    because the FEM run used the nominal MLP values. Only the compliance
    correction (substrate Poisson's ratio) varies with Cij uncertainty.

    Returns dict with keys: Er_plus_C11, Er_minus_C11, ... and delta_Er_total.
    """
    nominal = oliver_pharr(df_ph, theta_rad, epsilon,
                           UNLOAD_FIT_UPPER, UNLOAD_FIT_LOWER)
    S    = nominal['S']
    hc   = nominal['hc']
    Ac   = nominal['Ac']
    Er_0 = nominal['Er_MPa']

    results = {'Er_nominal_MPa': Er_0}
    delta_sq = 0.0

    for cname, C_nom, dC in [
        ('C11', C11_MPa, dC11 * 1000),   # GPa → MPa
        ('C12', C12_MPa, dC12 * 1000),
        ('C44', C44_MPa, dC44 * 1000),
    ]:
        for sign, tag in [(+1, 'plus'), (-1, 'minus')]:
            C11p = C11_MPa + sign * dC if cname == 'C11' else C11_MPa
            C12p = C12_MPa + sign * dC if cname == 'C12' else C12_MPa
            C44p = C44_MPa + sign * dC if cname == 'C44' else C44_MPa
            nu_p = hill_poisson(C11p, C12p, C44p)
            # Er unchanged (contact mechanics fixed by FEM output)
            # Only Es (substrate modulus) changes — propagate back to Er
            # Er = E_s / (1 - nu_s^2) corrected for indenter contribution
            # Full formula: 1/Er = (1-nu_s^2)/Es + (1-nu_i^2)/Ei
            # Here we perturb nu_s only; Es is taken from nominal inversion
            Es_nom = substrate_modulus(Er_0, hill_poisson(C11_MPa, C12_MPa, C44_MPa))
            Er_pert = 1.0 / (
                (1 - nu_p**2) / Es_nom
                + (1 - NU_IND**2) / E_IND
            )
            results[f'Er_{tag}_{cname}_MPa'] = Er_pert

        dEr = 0.5 * abs(
            results[f'Er_plus_{cname}_MPa'] - results[f'Er_minus_{cname}_MPa']
        )
        results[f'delta_Er_{cname}_MPa'] = dEr
        delta_sq += dEr**2

    results['delta_Er_total_MPa'] = np.sqrt(delta_sq)  # quadrature combination
    return results


if FEM_INPUTS_CSV.exists():
    fem_inputs = pd.read_csv(FEM_INPUTS_CSV)

    # Normalise composition column to match COMPOSITIONS keys
    # Expected format in CSV: 'fe16cr00', 'fe08cr08', 'fe04cr12', 'fe00cr16'
    # Map to: 'Fe16Cr0', 'Fe8Cr8', 'Fe4Cr12', 'Fe0Cr16'
    label_map = {
        'fe16cr00': 'Fe16Cr0',
        'fe08cr08': 'Fe8Cr8',
        'fe04cr12': 'Fe4Cr12',
        'fe00cr16': 'Fe0Cr16',
    }
    fem_inputs['label'] = fem_inputs['composition'].map(label_map)
    fem_inputs = fem_inputs.dropna(subset=['label']).set_index('label')

    # Run sensitivity for each composition
    uncertainty_rows = []
    for label, result in op_results.items():
        if label not in fem_inputs.index:
            print(f'{label}: not found in fem_material_inputs_best.csv — skipping')
            continue
        row   = fem_inputs.loc[label]
        cfg   = COMPOSITIONS[label]
        sens  = er_sensitivity(
            ph_data[label],
            cfg['C11'], cfg['C12'], cfg['C44'],
            row['std_C11_GPa'], row['std_C12_GPa'], row['std_C44_GPa'],
            THETA_RAD, EPSILON,
        )
        uncertainty_rows.append({
            'label'          : label,
            'cr_pct'         : cfg['cr_pct'],
            'Er_GPa'         : sens['Er_nominal_MPa'] * MPA_TO_GPA,
            'delta_Er_GPa'   : sens['delta_Er_total_MPa'] * MPA_TO_GPA,
            'delta_C11_GPa'  : row['std_C11_GPa'],
            'delta_C12_GPa'  : row['std_C12_GPa'],
            'delta_C44_GPa'  : row['std_C44_GPa'],
        })

    unc_df = pd.DataFrame(uncertainty_rows).set_index('label')

    # Plot Er vs Cr% with GP uncertainty bands
    fig, ax = plt.subplots(figsize=(7, 5))
    ax.errorbar(
        unc_df['cr_pct'],
        unc_df['Er_GPa'],
        yerr=unc_df['delta_Er_GPa'],
        fmt='o-', color='#1f4e79', lw=1.8, ms=7,
        capsize=5, elinewidth=1.2,
        label='Eᵣ ± GP σ (quadrature over C11, C12, C44)',
    )
    for _, row in unc_df.iterrows():
        ax.annotate(
            row.name,
            (row['cr_pct'], row['Er_GPa']),
            textcoords='offset points', xytext=(6, 4), fontsize=8
        )
    ax.set_xlabel('Cr content (at%)')
    ax.set_ylabel('Reduced modulus Eᵣ (GPa)')
    ax.set_title(
        'Eᵣ vs Cr% with GP surrogate uncertainty\n'
        'Fe-Cr BCC, conical FEM nanoindentation'
    )
    ax.legend()
    ax.set_xlim(-5, 105)
    plt.tight_layout()
    plt.savefig('../fem/post/fecr_Er_uncertainty.png', dpi=200, bbox_inches='tight')
    plt.show()

    display(unc_df)
    unc_df.to_csv('../fem/post/fecr_Er_uncertainty.csv')
    print('Saved: fecr_Er_uncertainty.csv')

else:
    print(
        f'File not found: {FEM_INPUTS_CSV}\n'
        'Expected at: ../analysis/fem_material_inputs_best.csv  (relative to notebooks/)'
    )


## 9. Export Results

In [ ]:
# Summary table
summary_df.to_csv('../fem/post/fecr_op_results.csv')
print('Saved: fecr_op_results.csv')

# Full P(h) curves for all compositions
all_ph = []
for label, df in ph_data.items():
    df_out = df[['time','h','P','phase']].copy()
    df_out.insert(0, 'composition', label)
    df_out.insert(1, 'cr_pct', COMPOSITIONS[label]['cr_pct'])
    all_ph.append(df_out)
pd.concat(all_ph, ignore_index=True).to_csv('../fem/post/fecr_ph_all.csv', index=False)
print('Saved: fecr_ph_all.csv')

print('\nSummary:')
display(summary_df)